### Why should you care?

Will I be asked to enumerate all 56 assignments?
   No. You might be asked why the state assignment matters (answer: it affects
   circui size after K-map minimisation) or to apply the two heuristic rules to
   pick a good assignment.

Will I be asked to find isomorphs?
   Maybe. The exam might show an assignment and ask you to write its isomorphs
   by inverting columns. That's just flipping 0 <-> 1 in each column -- 
   it's mechanical.

Will I need the heuristic rules?
   Yes -- L11 (traffic lights) uses them explicitly. 
      Rule 1: states with same next-state get adjacent codes.
      Rule 2: states reachable from the same state get adjacent code. 
              `Adjacent` = digger by 1 bit.

What's the big picture?
   State assignment in a design choice that doesn't affect correctness (any
   assignment works) but affects efficiency (some give smaller circuits).
   For exams: know why it matters, know the rules, don't memorise the 
   combinatorics.

   Yes, the FSM diagram (the circles and arrows) is almost always needed. While
   the math happens in the tables, the diagram is the bridge that stops you from
   making fatal logical errors when reading the English instructions.

   ... breakdown of how you execute a full design problem like this in an exam.


### Step 1: Formalise the Problem (Define the States)
   Before doing any math, you have to figure out how many states your system 
   needs. Because this is a MOORE MACHINE, the outputs (the lights) depend only
   on the current state. Therefore, every unique combination of lights require
   its own dedicaed state.

   Looking at the sequence, there are 6 unique light patterns.

   - The "Red" Trap: Notice that the pattern where both direction see "Red"
     occurs twice in the cycle. Can we save hardware and jsut use one state for
     both? No! One Red/Red state transitions into the North/South sequence, and
     the other transitions into the East/West sequence. Because they lead to 
     different futures, they must be mathematically separate states (State 1 and
     State 4).


### Step 2: The State Transition Table
   Now you can convert the circles and arrows into a table based on your input,
   J (Junction Type)

---

The big Idea: A register is just a row of D flip flops sitting side by side, 
each storing one bit. The interesting question is: what do you feed into each 
flip-flop's D input? That determines what the register does on the next
clock edge.

A MULTIPLEXER (MUX) is just a switch. A 4-way MUX has 4 possible inputs and 2 
control bits that pick which one passes through to the output. Think of it like
a  TV remote with 4 channels -- the 2-bit control selects which channel you 
watch. 

For the four-function shift register, each flip-flop gets its own 4-way MUX in
front of it. The 4 inputs to that MUX are:
   - Channel 00 (Hold): ...
   ...

All the MUXes share the same 2-bit control signal, so all flip-flops do the same 
operation simultaneously. Shift right - multiply by 2, shift left = divide by 2
(in binary).

The DECODER on slide 22 is the opposite of a MUX: it takes 2-bit input and 
activates exactl one of 4 output lines. Input 00 -> U0 = 1... It's used inside 
the MUX to route the signal. 




--------
--------


    n
----/--- Indicates a set of n wires (bus), one for each register bit. 

### The Problem: Selecting the Source (S7)
   To decide who "speaks" on the bus, we use a Multiplexer. Since we have 8
   registers, we need an 8-to-1 Multiplexer. 

   Slide 7 reintroduces the 4-to-1 multiplexor from previous lecuter, but adds 
   a crucial new feature: the ENABLE pin.
   - If `EN = 0`: The MUX is turned off. The output is forcefully pegged to `0`,
     no matter what the data or selection lines are doing.
   - If `EN = 1`: The MUX works normally. 

   ... a gatekeeper... in the context of SHARED COMPUTER BUS, its purpose goes 
   beyond jsut "pausing" a read/write--it prevents physical hardware collisions.

   ...

   1. A bus is a single set of shared electrical wires.
   2. If two registers try to "speak" (output their data) onto the shared wires
      at the exact same time, the voltages clash. This is a short...
      

The Problem: Loading the Destination (Slide 13)
   When data is on the common bus, every single register can "see" it. However,
   we only want one specific register to actually save that data.

   How do we trigger a register to save data? We send a pulse to its Clock input.
   Therefore, to select a destination register, we don't switch the data lines;
   we take the single master system clock pulse and route it to only one of the
   eight registers.

   To take one input (the clock pulse) and route it to one of many outputs. We use a 
   DEMULTIPLEXER. 

---

### What is a Comparator?
   A comparator is a fundamental logic circuit that takes two binary numbers
   (let's call them A and B) and tells you their relationship. It acts exactly
   like the comparison operators in programming (e.g., >, <, ==).

   In it's most complete form, a comparator has three distinct output wires. 
   
   - A > B (High if $A$ is strictly greater than $B$)
   - A < B (High if $A$ is strictly less than $B$)
   - A = B (High if $A$ and $B$ are identical)

   Comparators are the beating heart of CPU control flow. Every time you write 
   an `if`

---

   ... start doing math... looking at the fundamental building blocks of the 
   ALU (Arithmetic Logic Unit).

   Since the visualisers are ...


1. The Half Adder (Since 2, 3, 5)
   If ou want to add...


...

2. The Serial Adder (Slide 12)
   If you are incredibly constrained on space and don't want 32 parallel wires,
   you can use a Serial Adder.
   - Instead of 32 adders, you juse use ONE Full Adder,
   - You feed the numbers in sequentially (least significant bit first), one
     clock cycle at a time.
   - The genius part: You take the C_out and wire it into a D flip-flop 
     (the `D-Q` box). The flip-flop holds the carry for one clock cycle and then
     feeds it right back into the C_in for the next calculation. It "remembers"
     the carry for the next big pair!


3. Subtraction: Borrow and Payback 
   To subtract two numbers (A - B), we don't carry; we BORROW.

   The TT on S13 introduces a third input P (Payback, or Borrow In).
       ...


4. The Subtractor Circuit (Slide 15)
   Slide 15 derives the Boolean equations from that truth table.

   THE DIFFERENCE EQUATION:
   ... look familar? It is the exact same equation as the Full adder's SUM
      (Sum = A XOR B XOR C_in)

   THE BORROW EQUATION:
      The equation for the borrow out is different from the carry out:
         BORROW = B AND P + A' AND (B XOR P)

      The fact that the core `DIFFERENCE` logic is identical to the `SUM` logic
      is a huge hint for how CPU designers actually build ALUs. They don't 
      build separate adder and subtractor circuits; they reuse the same 
      hardware!